# Output Parsing

Language models output text. But there are times where you want to get more structured information than just text back.

Output parsers are classes that help structure language model responses. There are two main methods an output parser must implement:

- **Get format instructions**: A method which returns a string containing instructions for how the output of a language model should be formatted.
- **Parse**: A method which takes in a string (assumed to be the response from a language model) and parses it into some structure.

- Output Parsing
  - StrOutputParser
  - JsonOutputParser
  - CSV Output Parser
  - Datatime Output Parser
  - Structured Output Parser (Pydantic or Json)

## Pydantinc Output Parser

In [1]:
import  os
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)
from dotenv import load_dotenv

load_dotenv('./../../.env')

base_url = os.getenv('OLLAMA_BASE_URL')
model = os.getenv('OLLAMA_LLM_MODEL')

print(model)

llm = ChatOllama(base_url=base_url, model=model)

llama3.2


We are now creating a Pydantic model  to instruct the LLM how we want the output to be formatted.

In [2]:

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Joke(BaseModel):
  """Joke to tell user."""
  setup: str = Field(description="The setup of the joke")
  punchline: str = Field(description="The punchile of the joke")
  rating: int | None = Field(description="The rating of the joke", default=None)
  

And then we instantiate a `PydanticOutputParser`, which has a method `get_format_instructions` that will contain the command to the LLM output.

In [3]:
parser = PydanticOutputParser(pydantic_object=Joke)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Joke to tell user.", "properties": {"setup": {"description": "The setup of the joke", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchile of the joke", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "The rating of the joke", "title": "Rating"}}, "required": ["setup", "punchline"]}
```


In [4]:
from langchain_core.prompts import PromptTemplate


prompt = PromptTemplate(
    template='''
Answer the user query with a joke. Here is your formatting instruction.
{format_instruction}

Query: Tell me a joke about {joke_theme}
Answer:''',
    input_variables=['joke_theme'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = prompt | llm | parser

In [5]:
print(chain.invoke({'joke_theme': 'cats'}))

setup='Why did the cat join a band?' punchline='Because it wanted to be the purr-cussionist!' rating=None


### Important
The above `invoke` may not work in smaller models. The LLM simply ignores the requested formatted, causing a Pydantic error.

#### Tested in
- qwen3.5:2b <b>did not work</b>
- llama3.2 (3 billion) <b>it did work</b>

# Parsing with `.with_structured_output()` method

- Alternatively we can directly instruct the LLM to format the output properly, instead of passing it in the partial_variables.
- The `.with_structured_output()` method takes a schema as input which specifies the names, types, and descriptions of the desired output attributes.
- The schema can be specified as a TypedDict class, JSON Schema or a Pydantic class.

In [6]:
structed_llm = llm.with_structured_output(Joke)
chain = prompt | llm
result: Joke = chain.invoke({'joke_theme': 'dogs'}).content
print(result)

{"setup": "Why did the dog go to the vet?", "punchline": "Because he was feeling ruff!", "rating": null}


We can use it directly with the LLM, without creating a template.

In [7]:
structed_llm = llm.with_structured_output(Joke)
structed_llm.invoke('Tell me a joke about cats')

Joke(setup='Why did the cat join a band?', punchline='Because it wanted to be the purr-cussionist!', rating=None)

## Json Output Parser
We may also format the output as JSON, but still using the Pydantic to instruct the LLM on how we want the shape of our data.

In [9]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate


prompt = PromptTemplate(
    template='''
Answer the user query with a joke. Here is your formatting instruction.
{format_instruction}

Query: Tell me a joke about {joke_theme}
Answer:''',
    input_variables=['joke_theme'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)
json_output_parser = JsonOutputParser(pydantic_object=Joke)
chain = prompt | llm | json_output_parser
output = chain.invoke({'joke_theme': 'birds'})
print(output)

{'setup': 'Why did the bird go to the doctor?', 'punchline': 'Because it had a fowl cough!', 'rating': None}


## CSV Output Parser

- This output parser can be used when you want to return a list of comma-separated items.


In [10]:
# value1, values2, values3, so on

from langchain_core.output_parsers import CommaSeparatedListOutputParser

csv_parser = CommaSeparatedListOutputParser()

print(csv_parser.get_format_instructions())

prompt = PromptTemplate(
    template='''
    Answer the user query with a list of values. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': csv_parser.get_format_instructions()}
)

chain = prompt | llm | csv_parser

output = chain.invoke({
    'query': 'generate my website seo keywords. I have content about the NLP and LLM.'
})

print(output)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['natural language processing', 'large language models', 'AI', 'machine learning', 'deep learning', 'text analysis', 'sentiment analysis', 'language understanding', 'conversational AI', 'chatbots', 'semantic search', 'entity recognition', 'information extraction']
